# Generate a synthetic card dataset

Stage one predicts card boxes (COCO category: card). Use the grouped controls to select either reference deck and narrow appearance ranges. Card height is derived from width and a width/height ratio of 0.64–0.69. All shaded patterns are selected by default.

Scenes mix scattered, grid and clustered cards with small gaps and occasional slight overlap. Grid and cluster rotations use the layout jitter control; scattered cards use the full rotation range. Shared perspective acts on the entire scene. Masks describe visible card surfaces; corner keypoints retain the full card geometry, with covered corners marked as occluded.

Lighting controls add soft contact shadows and a brightness gradient. Training adds further illumination variation on the fly.

Glare spots straddle card corners to reduce edge contrast without changing the ground truth. Use the glare probability, strength and radius controls to tune difficulty.

At 1280×720, the default card-width range of 45–215 pixels covers far-away through close-up cards, based on the real annotations (short side roughly 5–23% of image short side), with a margin. Shared perspective and layout fitting change the final projected sizes. Scene widths are sampled logarithmically, with modest per-card jitter in scattered scenes. Grid and cluster cards share a scale and can be portrait or landscape.


## Backgrounds

Use `download_backgrounds.ipynb` to download and approve tabletop images. Choose `gray`, `real_scene`, or `mixed`; the real fraction applies to mixed scenes. Images are cropped without stretching. Source attribution is recorded in each generated image entry.

Generate separate training and validation runs using their respective background partitions, then set `val_data` explicitly in the training notebook. Splitting one generated run by image does not separate its source backgrounds. Review and approve usable tabletop surfaces: search results can contain furniture, objects, or unsuitable viewpoints.

Select `data/backgrounds/tabletop_textures` for the reviewed Poly Haven surface pool.


In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from set_master.generation.synthetic_dataset import DatasetConfig, generate_coco_dataset
from set_master.models import BackgroundType, BackgroundSplit, Color, Shading, ShadingPattern, Shape, SceneLayout
from set_master.paths import BACKGROUNDS_DIR, TABLETOP_TEXTURES_DIR, SYNTHETIC_DATASETS_DIR

defaults = DatasetConfig()
layout_orientations = widgets.SelectMultiple(
    options=[("Portrait cards", 0), ("Landscape cards", 90)],
    value=defaults.layout_orientations,
    description="Orientation",
)
card_scale_jitter = widgets.FloatSlider(
    value=defaults.card_scale_jitter, min=0, max=0.3, step=0.01, description="Scale jitter"
)
layouts = widgets.SelectMultiple(
    options=[(item.value.title(), item) for item in SceneLayout], value=defaults.layouts, description="Layouts"
)
scene_ranges = {}
for name in (
    "layout_gap",
    "scene_perspective",
    "shadow_opacity",
    "illumination_strength",
    "glare_strength",
    "glare_radius_ratio",
):
    scene_ranges[name] = widgets.FloatRangeSlider(
        value=getattr(defaults, name),
        min=0.01 if name == "glare_radius_ratio" else 0,
        max=0.6,
        step=0.01,
        description=name.replace("_", " "),
        continuous_update=False,
        style={"description_width": "180px"},
        layout=widgets.Layout(width="500px"),
    )
layout_rotation_jitter = widgets.FloatSlider(
    value=defaults.layout_rotation_jitter, min=0, max=30, step=1, description="Layout rotation"
)
overlap_probability = widgets.FloatSlider(
    value=defaults.overlap_probability, min=0, max=1, step=0.05, description="Overlap chance"
)
max_overlap_fraction = widgets.FloatSlider(
    value=defaults.max_overlap_fraction, min=0, max=0.15, step=0.01, description="Max overlap"
)
shadow_blur = widgets.IntSlider(value=defaults.shadow_blur, min=1, max=41, step=2, description="Shadow blur")
glare_probability = widgets.FloatSlider(
    value=defaults.glare_probability, min=0, max=1, step=0.05, description="Glare chance"
)
shadow_offset = widgets.IntRangeSlider(value=defaults.shadow_offset, min=0, max=20, description="Shadow offset")
patterns = widgets.SelectMultiple(
    options=[(item.value.replace("_", " ").title(), item) for item in ShadingPattern],
    value=defaults.shading_patterns,
    description="Patterns",
)
shapes = widgets.SelectMultiple(
    options=[(item.name.title(), item) for item in Shape], value=defaults.shapes, description="Shapes"
)
colors = widgets.SelectMultiple(
    options=[(item.name.title(), item) for item in Color], value=defaults.colors, description="Colors"
)
shadings = widgets.SelectMultiple(
    options=[(item.value.title(), item) for item in Shading], value=defaults.shadings, description="Shadings"
)
image_count = widgets.IntText(value=defaults.image_count, description="Images")
image_width = widgets.IntText(value=defaults.image_width, description="Image width")
image_height = widgets.IntText(value=defaults.image_height, description="Image height")
cards_per_image = widgets.IntRangeSlider(
    value=defaults.cards_per_image, min=0, max=20, description="Cards/image", continuous_update=False
)
rotation_degrees = widgets.FloatRangeSlider(
    value=defaults.rotation_degrees, min=-180, max=180, step=1, description="Rotation", continuous_update=False
)
skew_x_degrees = widgets.FloatRangeSlider(
    value=defaults.skew_x_degrees, min=-30, max=30, step=1, description="Skew X", continuous_update=False
)
skew_y_degrees = widgets.FloatRangeSlider(
    value=defaults.skew_y_degrees, min=-30, max=30, step=1, description="Skew Y", continuous_update=False
)
perspective_jitter = widgets.FloatRangeSlider(
    value=defaults.perspective_jitter, min=0, max=0.3, step=0.01, description="Perspective", continuous_update=False
)
background_type = widgets.Dropdown(
    options=[item.value for item in BackgroundType],
    value=defaults.background_type.value,
    description="Background type",
    style={"description_width": "140px"},
)
background_dir = widgets.Combobox(
    value=str(BACKGROUNDS_DIR),
    options=[str(BACKGROUNDS_DIR), str(TABLETOP_TEXTURES_DIR)],
    description="Background folder",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="650px"),
)
background_split = widgets.Dropdown(
    options=[item.value for item in BackgroundSplit],
    value=defaults.background_split.value,
    description="Background split",
    style={"description_width": "140px"},
)
real_background_probability = widgets.FloatSlider(
    value=defaults.real_background_probability,
    min=0,
    max=1,
    step=0.05,
    description="Real fraction",
    continuous_update=False,
)
background_gray = widgets.IntSlider(
    value=defaults.background_gray, min=0, max=255, description="Background", continuous_update=False
)
seed = widgets.IntText(value=defaults.seed, description="Seed")
output_root = widgets.Text(value=str(SYNTHETIC_DATASETS_DIR), description="Output folder")

# Keep appearance ranges within the modest variation observed in the reference decks.
range_widgets = {}
for name in (
    "card_width",
    "card_aspect_ratio",
    "feature_count",
    "feature_width_ratio",
    "feature_height_ratio",
    "symbol_spacing_ratio",
    "corner_radius_ratio",
    "stripe_spacing",
    "line_thickness",
    "squiggle_amplitude",
    "squiggle_waves",
    "squiggle_thickness",
):
    low, high = getattr(defaults, name)
    slider_type = widgets.IntRangeSlider if isinstance(low, int) else widgets.FloatRangeSlider
    range_widgets[name] = slider_type(
        value=(low, high),
        min=low,
        max=high,
        step=1 if isinstance(low, int) else 0.001,
        description=name.replace("_", " "),
        continuous_update=False,
        style={"description_width": "180px"},
        layout=widgets.Layout(width="500px"),
    )
scene_group = widgets.VBox(
    [
        image_count,
        image_width,
        image_height,
        cards_per_image,
        rotation_degrees,
        skew_x_degrees,
        skew_y_degrees,
        perspective_jitter,
        seed,
        output_root,
    ]
)
card_group = widgets.VBox(
    [shapes, colors, shadings]
    + [
        range_widgets[name]
        for name in (
            "card_width",
            "card_aspect_ratio",
            "feature_count",
            "feature_width_ratio",
            "feature_height_ratio",
            "symbol_spacing_ratio",
            "corner_radius_ratio",
        )
    ]
)
pattern_group = widgets.VBox([patterns, range_widgets["stripe_spacing"], range_widgets["line_thickness"]])
squiggle_group = widgets.VBox(
    [range_widgets[name] for name in ("squiggle_amplitude", "squiggle_waves", "squiggle_thickness")]
)
background_group = widgets.VBox(
    [background_type, background_dir, background_split, real_background_probability, background_gray]
)
layout_group = widgets.VBox(
    [
        layouts,
        layout_orientations,
        card_scale_jitter,
        scene_ranges["layout_gap"],
        layout_rotation_jitter,
        scene_ranges["scene_perspective"],
        overlap_probability,
        max_overlap_fraction,
    ]
)
lighting_group = widgets.VBox(
    [
        scene_ranges["shadow_opacity"],
        shadow_blur,
        shadow_offset,
        scene_ranges["illumination_strength"],
        glare_probability,
        scene_ranges["glare_strength"],
        scene_ranges["glare_radius_ratio"],
    ]
)
groups = widgets.Accordion(
    children=[scene_group, layout_group, lighting_group, background_group, card_group, pattern_group, squiggle_group]
)
for index, title in enumerate(
    [
        "Scene and output",
        "Layouts and shared perspective",
        "Lighting",
        "Backgrounds",
        "Card and symbols",
        "Shaded patterns",
        "Squiggle geometry",
    ]
):
    groups.set_title(index, title)
generate_button = widgets.Button(description="Generate dataset", button_style="primary")
status = widgets.Output()


def generate(_):
    with status:
        status.clear_output()
        try:
            config = DatasetConfig(
                output_root=Path(output_root.value),
                image_count=image_count.value,
                image_width=image_width.value,
                image_height=image_height.value,
                cards_per_image=tuple(cards_per_image.value),
                rotation_degrees=tuple(rotation_degrees.value),
                perspective_jitter=tuple(perspective_jitter.value),
                skew_x_degrees=tuple(skew_x_degrees.value),
                skew_y_degrees=tuple(skew_y_degrees.value),
                layout_orientations=tuple(layout_orientations.value),
                card_scale_jitter=card_scale_jitter.value,
                layouts=tuple(layouts.value),
                layout_rotation_jitter=layout_rotation_jitter.value,
                overlap_probability=overlap_probability.value,
                max_overlap_fraction=max_overlap_fraction.value,
                glare_probability=glare_probability.value,
                shadow_blur=shadow_blur.value,
                shadow_offset=tuple(shadow_offset.value),
                **{name: tuple(slider.value) for name, slider in scene_ranges.items()},
                background_gray=background_gray.value,
                seed=seed.value,
                background_type=BackgroundType(background_type.value),
                background_dir=Path(background_dir.value),
                background_split=BackgroundSplit(background_split.value),
                real_background_probability=real_background_probability.value,
                shading_patterns=tuple(patterns.value),
                shapes=tuple(shapes.value),
                colors=tuple(colors.value),
                shadings=tuple(shadings.value),
                **{name: tuple(slider.value) for name, slider in range_widgets.items()},
            )
        except ValueError as error:
            print(error)
            return
        print(f"Dataset saved to {generate_coco_dataset(config)}")


generate_button.on_click(generate)
display(widgets.VBox([groups, generate_button, status]))